# Test-set inference: lenient blocking → mDeBERTa cross-encoder (800k-bal) → submission TSVs

Everything runs as parallel SageMaker jobs that share one S3 work area:

| stage | where | jobs |
|---|---|---|
| **Lenient blocking** (`lenient_blocking.py`): any shared name word / squashed name / phonetic code / address word within the country → top 300 by IDF → rapidfuzz + jellyfish fuzzy score → keep **top 40 per S1, no threshold** (+ near-exact name/address hits up to 80) | CPU | 8 test jobs (India ×4, US ×3, France ×1) + 2 validation jobs (30k held-out train S1, India/US) |
| **Scoring** (`score_pairs.py`): every candidate pair through `sagemaker_cross_800k_bal` | GPU | 4 multi-GPU jobs draining a shared S3 shard queue; they start scoring as soon as the first shards land |
| **Submission** (`make_submission.py`): validation picks the best (model threshold, fuzzy floor) by macro F0.5, then applies it to test | this notebook | — |

Shards stream to S3 as they are written, and the GPU jobs pick them up straight away, so blocking and scoring overlap.
The 10 CPU and 4 GPU instances, plus the 1500k training job's `ml.g6.xlarge`, use all 15 instances the account allows for training jobs.

Inputs already in S3: `dataset/test/`, `dataset/train_sources/`, `dataset/train_gt/`, `dataset/val_s1_ids/`,
the adapter `jobs/output/amazon-ml-cross-800k-bal-20260927-191710/output/model.tar.gz`, and code under `code/business_entity_resolution/`.

Run the cells top to bottom. Step 4 can be re-run at any time and does not affect the jobs.

## 1. Setup

In [ ]:
import json, os, subprocess, tarfile, time
import boto3

REGION = 'ap-south-1'
BUCKET = 'sagemaker-ap-south-1-845989089398'
PREFIX = 'amazon-entity-resolution'
S3     = f's3://{BUCKET}/{PREFIX}'
ROLE   = 'arn:aws:iam::845989089398:role/service-role/AmazonSageMakerAdminIAMExecutionRole'
GPU_IMAGE = '763104351884.dkr.ecr.ap-south-1.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker'
CPU_IMAGE = '763104351884.dkr.ecr.ap-south-1.amazonaws.com/pytorch-training:2.8.0-cpu-py312-ubuntu22.04-sagemaker'
ADAPTER_S3 = f'{S3}/jobs/output/amazon-ml-cross-800k-bal-20260927-191710/output/model.tar.gz'

RUN_ID = 'infer-800kbal-v1'          # change to start a completely fresh run
RUN    = f'{S3}/runs/{RUN_ID}'        # work area: <RUN>/{test,val}/{pairs,s1_ids,done,scores}

# Blocking jobs: (tag, split, countries, part, parts, instance). One instance each.
BLOCK_JOBS = [
    ('test-india-0', 'test', 'India', 0, 4, 'ml.r6i.4xlarge'),
    ('test-india-1', 'test', 'India', 1, 4, 'ml.r6i.4xlarge'),
    ('test-india-2', 'test', 'India', 2, 4, 'ml.r6i.4xlarge'),
    ('test-india-3', 'test', 'India', 3, 4, 'ml.r6i.4xlarge'),
    ('test-us-0',    'test', 'US',    0, 3, 'ml.m5.4xlarge'),
    ('test-us-1',    'test', 'US',    1, 3, 'ml.m5.4xlarge'),
    ('test-us-2',    'test', 'US',    2, 3, 'ml.m5.4xlarge'),
    ('test-france',  'test', 'France', 0, 1, 'ml.m5.4xlarge'),
    ('val-india',    'val',  'India', 0, 1, 'ml.m6i.4xlarge'),
    ('val-us',       'val',  'US',    0, 1, 'ml.m6i.4xlarge'),
]
# Scoring jobs: every GPU of every instance drains the same queue (val first, then test).
# The account quota is 1 instance per type; any job that cannot get capacity fails harmlessly.
GPU_JOBS = ['ml.g6e.12xlarge', 'ml.g5.12xlarge', 'ml.g6.12xlarge', 'ml.g4dn.12xlarge']
N_DONE = {s: sum(1 for j in BLOCK_JOBS if j[1] == s) for s in ('val', 'test')}

ROOT = '/home/sagemaker-user/student_resource' if os.path.isdir('/home/sagemaker-user') else os.path.abspath('.')
CODE = f'{ROOT}/code/business_entity_resolution'
WORK = f'{ROOT}/work/{RUN_ID}'
OUTPUT = f'{ROOT}/output'
os.makedirs(WORK, exist_ok=True); os.makedirs(OUTPUT, exist_ok=True)

sm, s3, logs = (boto3.client(s, region_name=REGION) for s in ('sagemaker', 's3', 'logs'))

def sh(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)

def s3_count(prefix, suffix=''):
    b, _, k = prefix[5:].partition('/')
    n = 0
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=b, Prefix=k):
        n += sum(o['Key'].endswith(suffix) for o in page.get('Contents', []))
    return n

## 2. Pull code from S3, check inputs, bundle the job source

In [ ]:
sh(f'aws s3 sync {S3}/code/business_entity_resolution/ {CODE}/ --exclude "*__pycache__*" --only-show-errors')
sh(f'aws s3 cp {S3}/code/utils/validate_submission.py {ROOT}/utils/validate_submission.py --only-show-errors')
sh('pip install -q polars==1.44.2 pyarrow')

for p in ['dataset/test/test_source1.tsv', 'dataset/test/test_source2.tsv', 'dataset/test/test_source3.tsv',
          'dataset/train_sources/train_source1.tsv', 'dataset/train_sources/train_source2.tsv',
          'dataset/train_sources/train_source3.tsv', 'dataset/train_gt/train_ground_truth.tsv',
          'dataset/val_s1_ids/val_s1_ids_30k.txt',
          'jobs/output/amazon-ml-cross-800k-bal-20260927-191710/output/model.tar.gz']:
    ok = s3_count(f'{S3}/{p}') > 0
    print('OK     ' if ok else 'MISSING', p)
    assert ok, f'missing input {p}'

bundle = '/tmp/infer_sourcedir.tar.gz'
with tarfile.open(bundle, 'w:gz') as t:
    t.add(f'{CODE}/sagemaker_job/infer_entry.py', arcname='infer_entry.py')
    t.add(f'{CODE}/sagemaker_job/requirements-infer.txt', arcname='requirements.txt')
    for f in ('lenient_blocking.py', 'blocking.py', 'fuzzy.py', 'normalize.py', 'phonetic.py', 'score_pairs.py'):
        t.add(f'{CODE}/src/{f}', arcname=f)
SOURCE = f'{RUN}/source/sourcedir.tar.gz'
sh(f'aws s3 cp {bundle} {SOURCE} --only-show-errors')

## 3. Launch every job in parallel
Idempotent: a job name that already exists for this `RUN_ID` is not launched again.

In [ ]:
def job_name(tag):
    return f'er-{RUN_ID}-{tag}'[:63]

def channel(name, uri):
    return {'ChannelName': name, 'DataSource': {'S3DataSource': {
        'S3DataType': 'S3Prefix', 'S3Uri': uri, 'S3DataDistributionType': 'FullyReplicated'}}}

def launch(tag, image, instance, hp, channels, volume=50):
    name = job_name(tag)
    try:
        sm.describe_training_job(TrainingJobName=name); print('exists  ', name); return name
    except sm.exceptions.ClientError:
        pass
    sm.create_training_job(
        TrainingJobName=name, RoleArn=ROLE,
        AlgorithmSpecification={'TrainingImage': image, 'TrainingInputMode': 'File'},
        HyperParameters={'sagemaker_program': json.dumps('infer_entry.py'),
                         'sagemaker_submit_directory': json.dumps(SOURCE),
                         'sagemaker_region': json.dumps(REGION),
                         **{k: json.dumps(v) if isinstance(v, str) else str(v) for k, v in hp.items()}},
        InputDataConfig=channels,
        OutputDataConfig={'S3OutputPath': f'{RUN}/jobs/'},
        ResourceConfig={'InstanceType': instance, 'InstanceCount': 1, 'VolumeSizeInGB': volume},
        StoppingCondition={'MaxRuntimeInSeconds': 4 * 3600},
        Environment={'HF_HUB_DISABLE_TELEMETRY': '1', 'TOKENIZERS_PARALLELISM': 'false'})
    print('launched', name, instance); return name

JOBS = {}
for tag, split, country, part, parts, inst in BLOCK_JOBS:
    if split == 'test':
        chans = [channel('data', f'{S3}/dataset/test/')]
    else:
        chans = [channel('data', f'{S3}/dataset/train_sources/'), channel('s1ids', f'{S3}/dataset/val_s1_ids/')]
    JOBS[tag] = launch(tag, CPU_IMAGE, inst,
                       {'mode': 'block', 'out-s3': f'{RUN}/{split}', 'tag': tag, 'countries': country,
                        'part': str(part), 'parts': str(parts), 'k': '40', 'k-max': '80', 'top-m': '300'},
                       chans)

for inst in GPU_JOBS:
    tag = 'score-' + inst.split('.', 1)[1].replace('.', '-')
    try:
        JOBS[tag] = launch(tag, GPU_IMAGE, inst,
                           {'mode': 'score', 'out-s3': f'{RUN}/val,{RUN}/test',
                            'expect-done': f"{N_DONE['val']},{N_DONE['test']}", 'batch': '256'},
                           [channel('adapter', ADAPTER_S3)], volume=30)
    except sm.exceptions.ClientError as e:   # quota / capacity: the other GPU jobs still drain the queue
        print('could not launch', inst, '->', e)

## 4. Watch progress (re-run any time)
Shows job states and shard counts. It stops when every scoring job has ended. Interrupting this cell does not stop the jobs.

In [ ]:
def status():
    rows = []
    for tag, name in JOBS.items():
        d = sm.describe_training_job(TrainingJobName=name)
        rows.append((tag, d['TrainingJobStatus'], d['SecondaryStatus'], d.get('FailureReason', '')[:90]))
    return rows

while True:
    rows = status()
    print(time.strftime('%H:%M:%S'),
          ' | '.join(f"{sp}: pairs={s3_count(f'{RUN}/{sp}/pairs/', '.parquet')} "
                     f"scored={s3_count(f'{RUN}/{sp}/scores/', '.parquet')} "
                     f"done={s3_count(f'{RUN}/{sp}/done/')}/{N_DONE[sp]}" for sp in ('val', 'test')))
    for r in rows:
        if r[1] != 'Completed':
            print('   ', *r)
    if any(r[1] == 'Failed' and not r[0].startswith('score') for r in rows):
        print('!! a blocking job FAILED: scoring jobs would wait for it forever. Fix it and re-launch that tag, '
              'or run the stop cell at the bottom.')
    scorers = [r for r in rows if r[0].startswith('score')]
    if scorers and all(r[1] in ('Completed', 'Failed', 'Stopped') for r in scorers):
        break
    time.sleep(60)

## 5. Validation metrics → choose threshold (30k held-out train S1 vs the full train S2+S3 pool)

In [ ]:
V = f'{WORK}/val'
sh(f'aws s3 sync {RUN}/val/scores/ {V}/scores/ --exclude "*.claim" --only-show-errors')
sh(f'aws s3 sync {RUN}/val/s1_ids/ {V}/s1_ids/ --only-show-errors')
sh(f'aws s3 cp {S3}/dataset/train_gt/train_ground_truth.tsv {WORK}/train_ground_truth.tsv --only-show-errors')
sh(f'cd {CODE}/src && python make_submission.py --work {V} --out {V}/out --gt {WORK}/train_ground_truth.tsv '
   f'--threshold auto --metrics {V}/metrics.json')
M = json.load(open(f'{V}/metrics.json'))
BEST_T, BEST_F = M['best']['threshold'], M['best']['min_fuzzy']
print('\nblocking:'); [print(f'  {k:<7}', v) for k, v in M['blocking'].items() if k.startswith('top')]
print(f"\nchosen: model score >= {BEST_T}, fuzzy_score >= {BEST_F}")
print(json.dumps({k: M['best'][k] for k in ('macro_f0_5', 'precision', 'recall', 'singleton_accuracy',
                                            'macro_f0_5_non_singletons')}, indent=2))

## 6. Test submission: `output/matching_results.tsv` + `output/candidate_pairs.tsv`

In [ ]:
T = f'{WORK}/test'
sh(f'aws s3 sync {RUN}/test/scores/ {T}/scores/ --exclude "*.claim" --only-show-errors')
sh(f'aws s3 sync {RUN}/test/s1_ids/ {T}/s1_ids/ --only-show-errors')
n_pairs = s3_count(f'{RUN}/test/pairs/', '.parquet')
n_scores = len([f for f in os.listdir(f'{T}/scores') if f.endswith('.parquet')])
assert n_scores == n_pairs, f'only {n_scores}/{n_pairs} test shards scored: re-run step 4'

os.makedirs(f'{ROOT}/dataset/test', exist_ok=True)
if not os.path.exists(f'{ROOT}/dataset/test/test_source1.tsv'):
    sh(f'aws s3 cp {S3}/dataset/test/test_source1.tsv {ROOT}/dataset/test/test_source1.tsv --only-show-errors')

# --s1-file: one row per test S1 entity (empty list when no candidates / no match).
sh(f'cd {CODE}/src && python make_submission.py --work {T} --out {OUTPUT} '
   f'--threshold {BEST_T} --min-fuzzy {BEST_F} --s1-file {ROOT}/dataset/test/test_source1.tsv '
   f'--metrics {T}/output_stats.json')
sh(f'cd {ROOT} && python3 utils/validate_submission.py --matching output/matching_results.tsv '
   f'--candidate output/candidate_pairs.tsv --test-dir dataset/test')

sh(f'aws s3 cp {OUTPUT}/matching_results.tsv {RUN}/output/matching_results.tsv --only-show-errors')
sh(f'aws s3 cp {OUTPUT}/candidate_pairs.tsv {RUN}/output/candidate_pairs.tsv --only-show-errors')
sh(f'aws s3 cp {V}/metrics.json {RUN}/output/val_metrics.json --only-show-errors')
print(json.load(open(f'{T}/output_stats.json'))['output'])
print('\nUpload to the portal:', f'{OUTPUT}/matching_results.tsv', '| also in', f'{RUN}/output/')

## (Only if needed) stop every job of this run

In [ ]:
for tag, name in JOBS.items():
    if sm.describe_training_job(TrainingJobName=name)['TrainingJobStatus'] == 'InProgress':
        sm.stop_training_job(TrainingJobName=name); print('stopping', name)